# 18 — Identity, Authorization & Tool Security

## Learning requirements
Phân biệt các principal:
- end user;
- application/service;
- agent runtime;
- subagent;
- MCP server/tool;
- external API.

Phải hiểu:
- authentication vs authorization;
- RBAC vs ABAC;
- least privilege;
- delegated credentials;
- short-lived/scoped credentials;
- resource-level authorization;
- confused deputy problem;
- audit attribution: action này do ai yêu cầu và principal nào thực thi.

**Model không phải identity provider và không phải policy engine.**

## Identity chain

```text
User u123
  -> authenticated request
  -> Agent runtime (service identity)
  -> Tool call carrying user/tenant context
  -> Deterministic authorizer
  -> Scoped backend credential
  -> Resource
```

Không dùng một global admin credential cho mọi user chỉ vì agent cần nhiều tools.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Principal:
    user_id: str
    tenant_id: str
    roles: frozenset[str]

@dataclass(frozen=True)
class Resource:
    resource_id: str
    tenant_id: str
    owner_id: str

def can_read_project(principal: Principal, resource: Resource) -> bool:
    if principal.tenant_id != resource.tenant_id:
        return False
    return principal.user_id == resource.owner_id or "project_admin" in principal.roles

def can_delete_project(principal: Principal, resource: Resource) -> bool:
    if principal.tenant_id != resource.tenant_id:
        return False
    return "project_admin" in principal.roles

alice = Principal("u1", "t1", frozenset({"viewer"}))
admin = Principal("u2", "t1", frozenset({"project_admin"}))
foreign_admin = Principal("u3", "t2", frozenset({"project_admin"}))
project = Resource("p1", "t1", "u1")

assert can_read_project(alice, project)
assert not can_delete_project(alice, project)
assert can_delete_project(admin, project)
assert not can_delete_project(foreign_admin, project)

## Tool security rules

Mỗi tool production phải document:
1. caller/principal expected;
2. resources accessed;
3. permission required;
4. read vs write vs irreversible;
5. argument validation;
6. credential scope;
7. timeout/rate limit;
8. audit fields;
9. idempotency strategy;
10. HITL requirement.

Tool description giúp model chọn tool nhưng **không phải enforcement mechanism**.

## Exercise — Delegation design

Cho Interview Supervisor gọi Scanner Agent và Git/MCP tools. Thiết kế sao cho:
- scanner chỉ có read access repository;
- interviewer không có shell access;
- publisher chỉ được gọi sau approval;
- subagent không tự mở rộng permission;
- cross-tenant requests fail deterministically;
- credential không xuất hiện trong model context.

## Required output
- `artifacts/security/identity-model.md`
- `artifacts/security/tool-permission-matrix.md`
- tests cho owner/admin/cross-tenant/unknown-role cases.

## Done criteria
- Principle of least privilege áp dụng ở tool/resource level.
- Có attribution từ user request tới final side effect.
- Agent/subagent không được tự cấp quyền cho chính nó.
- Privileged operations có explicit approval + idempotency design.